# 01B — Gop DataMart Master & Kiem Tra Toan Dien

**Muc tieu:** Doc 3 file DataMart da ETL rieng le -> concat -> kiem tra QA -> luu Master.

> **Thu tu chay bat buoc:**
> 1. `01B_ETL_OULAD.ipynb`       -> tao `oulad_datamart.csv`
> 2. `01C_ETL_UCI_Dropout.ipynb` -> tao `uci_dropout_datamart.csv`
> 3. `01D_ETL_UCI_Perf.ipynb`    -> tao `uci_perf_datamart.csv`
> 4. **Notebook nay**            -> tao `DataMart_Student_Risk_Flat_Master.csv`

---

### Nguyen tac OOD (khong duoc vi pham)
- Cot LMS cua UCI_* phai la `NaN` (KHONG dien median cua OULAD)
- `admission_score_norm` phai co phuong sai > 0 o ca 3 nguon
- GPA su dung Piecewise Alignment: nguong qua mon = **2.0** nhat quan


In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.4f}'.format)

PROCESSED_DIR = 'data/processed/' if os.path.basename(os.getcwd()) != 'notebooks' else '../data/processed/'
print(f"PROCESSED_DIR: {os.path.abspath(PROCESSED_DIR)}")


## 1. Doc 3 File DataMart Da ETL

In [ ]:
# Danh sach file can load
source_files = {
    'OULAD'       : PROCESSED_DIR + 'oulad_datamart.csv',
    'UCI_Dropout' : PROCESSED_DIR + 'uci_dropout_datamart.csv',
    'UCI_Perf'    : PROCESSED_DIR + 'uci_perf_datamart.csv',
}

dfs = {}
missing = []

for name, fpath in source_files.items():
    if not os.path.exists(fpath):
        print(f"MISSING: {fpath}")
        print(f"  => Hay chay notebook ETL tuong ung truoc!")
        missing.append(name)
    else:
        dfs[name] = pd.read_csv(fpath)
        size_kb = os.path.getsize(fpath) / 1024
        print(f"OK  {name:<15} | shape: {str(dfs[name].shape):<15} | {size_kb:.1f} KB")

if missing:
    raise RuntimeError(f"Thieu {len(missing)} file: {missing}. Chay cac notebook ETL truoc!")

print()
print(f"Da load du 3 nguon. San sang de gop.")


## 2. Kiem Tra Cau Truc Truoc Khi Gop

In [ ]:
# Kiem tra cac cot phai nhat quan giua 3 file
cols_per_source = {name: list(df.columns) for name, df in dfs.items()}
ref_cols = cols_per_source['OULAD']

print("=== So sanh cau truc cot giua 3 nguon ===")
all_match = True
for name, cols in cols_per_source.items():
    if cols == ref_cols:
        print(f"  {name}: MATCH ({len(cols)} cols)")
    else:
        diff = set(ref_cols) ^ set(cols)
        print(f"  {name}: KHAC BIET! Cac cot sai lech: {diff}")
        all_match = False

print()
if all_match:
    print("Tat ca 3 nguon co cau truc cot giong nhau - san sang gop.")
else:
    print("Co su khac biet cot - kiem tra lai cac notebook ETL!")
    
print()
print("22 cot DataMart:")
for i, col in enumerate(ref_cols, 1):
    print(f"  {i:2d}. {col}")


## 3. Gop Master DataFrame

In [ ]:
df_master = pd.concat(list(dfs.values()), ignore_index=True)

print(f"=== MASTER DATAMART ===")
print(f"Shape: {df_master.shape}")
print()
print("--- Phan phoi theo nguon ---")
print(df_master['dataset_source'].value_counts())
print()
print("--- Phan phoi nhan rui ro (toan bo) ---")
print(df_master['target_risk_name'].value_counts())
print()
print("--- Ti le nhan theo tung nguon (%) ---")
pivot = df_master.groupby(['dataset_source', 'target_risk_name']).size()
pivot_pct = pivot.groupby(level=0).transform(lambda x: (x/x.sum()*100).round(1))
print(pivot_pct.unstack(fill_value=0))


## 4. Kiem Tra Chat Luong QA

In [ ]:
print("=== QA 1: Kiem tra NaN cot LMS theo nguon ===")
print("(OULAD phai = 0%, UCI phai = 100%)")
print()

lms_cols = ['lms_total_clicks', 'lms_active_days', 'lms_late_submits', 'lms_avg_quiz']
results = []
all_ok = True

for col in lms_cols:
    for src in ['OULAD', 'UCI_Dropout', 'UCI_Perf']:
        sub = df_master[df_master['dataset_source'] == src][col]
        pct_nan = sub.isna().mean() * 100
        expected_nan = 0.0 if src == 'OULAD' else 100.0
        ok = abs(pct_nan - expected_nan) < 0.01
        if not ok:
            all_ok = False
        status = "OK" if ok else "FAILED <=="
        results.append({'col': col, 'source': src, 'nan_pct': pct_nan, 'expected': expected_nan, 'status': status})
        print(f"  {col:<22} | {src:<12} | NaN={pct_nan:5.1f}% | Expected={expected_nan:.0f}% | [{status}]")

print()
if all_ok:
    print("Tat ca cot LMS NaN dung nhu ky vong.")
else:
    print("CO VI PHAM! Kiem tra lai cac notebook ETL.")


In [ ]:
print("=== QA 2: Kiem tra NaN cot training_point_norm ===")
print("(OULAD va UCI_Dropout phai = NaN, UCI_Perf phai co gia tri thuc)")
print()

for src in ['OULAD', 'UCI_Dropout', 'UCI_Perf']:
    sub = df_master[df_master['dataset_source'] == src]['training_point_norm']
    pct_nan = sub.isna().mean() * 100
    if src in ['OULAD', 'UCI_Dropout']:
        expected = "100% NaN"
        ok = pct_nan == 100.0
    else:
        expected = "0% NaN (co proxy absences)"
        ok = pct_nan == 0.0
    status = "OK" if ok else "FAILED <=="
    print(f"  {src:<12}: NaN={pct_nan:.1f}% | Expected: {expected} | [{status}]")


In [ ]:
print("=== QA 3: Phan phoi GPA sau Piecewise Alignment ===")
print("Nguong qua mon = GPA 2.0 phai nhat quan o ca 3 nguon")
print()

for src in ['OULAD', 'UCI_Dropout', 'UCI_Perf']:
    sub = df_master[df_master['dataset_source'] == src]['gpa_current_semester']
    pct_pass = (sub >= 2.0).mean() * 100
    print(f"[{src}]")
    print(f"  Mean={sub.mean():.3f} | Median={sub.median():.3f} | Std={sub.std():.3f}")
    print(f"  GPA in [0,4]: {sub.between(0,4).all()} | % GPA >= 2.0: {pct_pass:.1f}%")
    print()


In [ ]:
print("=== QA 4: Phuong sai admission_score_norm theo nguon ===")
print("Bat buoc phuong sai > 0 o ca 3 nguon (neu = 0 thi bien chet)")
print()

all_ok = True
for src in ['OULAD', 'UCI_Dropout', 'UCI_Perf']:
    sub = df_master[df_master['dataset_source'] == src]['admission_score_norm']
    var = sub.var()
    ok = var > 0
    if not ok:
        all_ok = False
    status = "OK" if ok else "FAILED - BIEN CHET! <=="
    print(f"  {src:<12}: var={var:.4f} | [{status}]")
    if ok:
        print(f"             min={sub.min():.3f} | max={sub.max():.3f} | mean={sub.mean():.3f}")

print()
if all_ok:
    print("Tat ca bien admission_score_norm co phuong sai > 0.")


In [ ]:
print("=== QA 5: Kiem tra student_code khong trung lap ===")
total = len(df_master)
unique = df_master['student_code'].nunique()
print(f"  Tong dong     : {total}")
print(f"  student_code unique: {unique}")
if total == unique:
    print("  Ket qua: Khong co student_code trung lap. [OK]")
else:
    dup_count = total - unique
    print(f"  CANH BAO: Co {dup_count} dong bi trung lap student_code!")
    
print()
print("=== Prefix phan biet nguon ===")
for src in ['OULAD', 'UCI_Dropout', 'UCI_Perf']:
    sample = df_master[df_master['dataset_source'] == src]['student_code'].iloc[0]
    print(f"  {src:<12}: vi du '{sample}'")


## 5. Luu File Master

In [ ]:
out_path = PROCESSED_DIR + 'DataMart_Student_Risk_Flat_Master.csv'
df_master.to_csv(out_path, index=False)

size_mb = os.path.getsize(out_path) / 1024 / 1024
print(f"Da luu: {os.path.abspath(out_path)}")
print(f"Shape : {df_master.shape}")
print(f"Size  : {size_mb:.2f} MB")
print()
print("Phan phoi nguon trong file Master:")
print(df_master['dataset_source'].value_counts())


## 6. Checklist QA Cuoi Cung

| # | Kiem tra | Ket qua |
|---|---|---|
| 1 | 3 nguon co mat: OULAD, UCI_Dropout, UCI_Perf | Kiem tra Cell 3 |
| 2 | 22 cot nhat quan giua 3 nguon | Kiem tra Cell 5 |
| 3 | LMS cua UCI_* = 100% NaN | Cell 9 |
| 4 | training_point_norm NaN dung nhu ky vong | Cell 10 |
| 5 | GPA nam trong [0,4], nguong qua mon = 2.0 | Cell 11 |
| 6 | admission_score_norm co phuong sai > 0 o ca 3 nguon | Cell 12 |
| 7 | student_code khong trung lap | Cell 13 |

> **Buoc tiep theo:** Mo `02_modeling.ipynb` — Huan luyen 5 mo hinh theo khung OOD.
>
> Luu y voi Random Forest & Logistic Regression: su dung `SimpleImputer(strategy='median')`
> trong sklearn `Pipeline` de xu ly cac cot NaN. Fit imputer CHI TREN tap OULAD Train,
> sau do transform ca Train va Test.
